In [ ]:
# import subprocess, sys

# subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
#                 'ultralytics', 'roboflow', 'supervision'], check=True)
!pip install -q ultralytics roboflow supervision

import torch
print(f'PyTorch:       {torch.__version__}')
print(f'CUDA:          {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:           {torch.cuda.get_device_name(0)}')
    print(f'VRAM:          {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from ultralytics import YOLO, checks
checks()

In [ ]:
import os
from roboflow import Roboflow
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
API_KEY = user_secrets.get_secret('ROBOFLOW')
rf = Roboflow(api_key=API_KEY)

DATASETS = [
    # (workspace, project, version, local_dir)
    ('wonkeun-jung-vfcwn', 'ingredients-agbcq',             1,  'ds1'),
    ('basket-z7ppw', 'ingredients-detection-xmjrt',                3,  'ds2'),
    ('yaman-e', 'food-ingredients-detection-qfit7',          50, 'ds3'),
    ('karel-cornelis-q2qqg', 'aicook-lcv4d',                1,  'ds4'),
]

for ws, proj, ver, loc in DATASETS:
    path = f'/kaggle/working/datasets/{loc}'
    print(f'Скачиваем {loc} ({proj})...')
    rf.workspace(ws).project(proj).version(ver).download('yolo26', location=path)

print('\nВсе датасеты скачаны!')
for _, _, _, loc in DATASETS:
    p = f'/kaggle/working/datasets/{loc}'
    yamls = [f for f in os.listdir(p) if f.endswith('.yaml')]
    print(f'  {loc}: {yamls}')


In [ ]:
import yaml
import json
from pathlib import Path

# ============================================================
# Фиксированный список целевых классов (68 штук)
# Только эти классы попадут в итоговый датасет.
# ============================================================
TARGET_CLASSES = ["apple", "avocado", "bacon", "banana", "beef", "bell_pepper", "blueberry", "bread", "broccoli", "butter", "cabbage", "carrot", "cauliflower", "celery", "cheese", "chicken", "chili_pepper", "coconut", "coriander", "cream", "cucumber", "egg", "eggplant", "fish", "flour_grain", "garlic", "ginger", "gourd", "grape", "leafy_greens", "lemon", "mango", "melon", "milk", "mushroom", "noodles", "nuts", "oil", "olive", "onion", "orange", "peach", "pear", "peas", "pineapple", "pomegrante", "pork", "potato", "pudding", "pumpkin", "raddish", "rice", "shrimp", "spinach", "strawberry", "sugar", "tomato"]
TARGET_SET = set(TARGET_CLASSES)

# ============================================================
# Таблица синонимов: произвольное_имя → TARGET_CLASS
# Нормализация: lower + пробелы/дефисы → '_'
# Если имя не найдено в таблице — класс ДРОПАЕТСЯ.
# ============================================================
SYNONYMS = {
    # ── apple ────────────────────────────────────────────
    'apple':             'apple',
    'apples':            'apple',
    'green_apple':       'apple',
    'greenapple':        'apple',
    'red_apple':         'apple',

    # ── avocado ──────────────────────────────────────────
    'avocado':           'avocado',
    'avocados':          'avocado',

    # ── bacon ────────────────────────────────────────────
    'bacon':             'bacon',

    # ── banana ───────────────────────────────────────────
    'banana':            'banana',
    'bananas':           'banana',

    # ── beef ─────────────────────────────────────────────
    'beef':              'beef',
    'meat':              'beef',
    'steak':             'beef',
    'ground_beef':       'beef',
    'ground_meat':       'beef',
    'minced_meat':       'beef',
    'buff_meat':         'beef',
    'mutton':            'beef',
    'lamb':              'beef',

    # ── bell_pepper ──────────────────────────────────────
    'bell_pepper':       'bell_pepper',
    'bell pepper':       'bell_pepper',
    'capsicum':          'bell_pepper',
    'sweet_pepper':      'bell_pepper',
    'sweet pepper':      'bell_pepper',
    'pepper':            'bell_pepper',

    # ── blueberry ────────────────────────────────────────
    'blueberry':         'blueberry',
    'blueberries':       'blueberry',
    'blue_berry':        'blueberry',

    # ── bread ────────────────────────────────────────────
    'bread':             'bread',
    'bagel':             'bread',
    'hash_brown':        'bread',
    'toast':             'bread',
    'pita':              'bread',
    'tortilla':          'bread',
    'bun':               'bread',

    # ── broccoli ─────────────────────────────────────────
    'broccoli':          'broccoli',

    # ── butter ───────────────────────────────────────────
    'butter':            'butter',
    'margarine':         'butter',

    # ── cabbage ──────────────────────────────────────────
    'cabbage':           'cabbage',
    'bok_choy':          'cabbage',
    'napa_cabbage':      'cabbage',

    # ── carrot ───────────────────────────────────────────
    'carrot':            'carrot',
    'carrots':           'carrot',

    # ── cauliflower ──────────────────────────────────────
    'cauliflower':       'cauliflower',

    # ── celery ───────────────────────────────────────────
    'celery':            'celery',
    'salary':            'celery',
    'celeriac':          'celery',

    # ── cheese ───────────────────────────────────────────
    'cheese':            'cheese',
    'goat_cheese':       'cheese',
    'feta':              'cheese',
    'mozzarella':        'cheese',
    'parmesan':          'cheese',
    'quark':             'cheese',

    # ── chicken ──────────────────────────────────────────
    'chicken':           'chicken',
    'chicken_breast':    'chicken',
    'chicken breast':    'chicken',
    'chicken_wing':      'chicken',
    'chicken_thigh':     'chicken',
    'chicken_gizzards':  'chicken',
    'turkey':            'chicken',
    'poultry':           'chicken',

    # ── chili_pepper ─────────────────────────────────────
    'chili_pepper':      'chili_pepper',
    'chili pepper':      'chili_pepper',
    'chilli':            'chili_pepper',
    'chili':             'chili_pepper',
    'hot_pepper':        'chili_pepper',
    'hot pepper':        'chili_pepper',
    'jalapeno':          'chili_pepper',
    'green_pepper':      'chili_pepper',
    'khursani':          'chili_pepper',
    'long_pepper':       'chili_pepper',
    'small_pepper':      'chili_pepper',

    # ── chocolate ────────────────────────────────────────
    'chocolate':         'chocolate',
    'cocoa':             'chocolate',

    # ── coconut ──────────────────────────────────────────
    'coconut':           'coconut',

    # ── coriander ────────────────────────────────────────
    'coriander':         'coriander',
    'cilantro':          'coriander',
    'dhaniya':           'coriander',

    # ── cream ────────────────────────────────────────────
    'cream':             'cream',
    'heavy_cream':       'cream',
    'whipping_cream':    'cream',

    # ── cucumber ─────────────────────────────────────────
    'cucumber':          'cucumber',
    'cucumbers':         'cucumber',

    # ── egg ──────────────────────────────────────────────
    'egg':               'egg',
    'eggs':              'egg',

    # ── eggplant ─────────────────────────────────────────
    'eggplant':          'eggplant',
    'aubergine':         'eggplant',
    'brinjal':           'eggplant',
    'green_brinjal':     'eggplant',

    # ── fish ─────────────────────────────────────────────
    'fish':              'fish',
    'salmon':            'fish',
    'tuna':              'fish',
    'cod':               'fish',
    'tilapia':           'fish',
    'lobster_tail':      'fish',
    'crab':              'fish',
    'oysters':           'fish',
    'oyster':            'fish',

    # ── flour_grain ──────────────────────────────────────
    'flour':             'flour_grain',
    'flour_grain':       'flour_grain',
    'wheat':             'flour_grain',
    'wheat_flour':       'flour_grain',
    'corn_flour':        'flour_grain',
    'oats':              'flour_grain',
    'bulgur':            'flour_grain',
    'grain':             'flour_grain',

    # ── garlic ───────────────────────────────────────────
    'garlic':            'garlic',
    'garlic_clove':      'garlic',

    # ── ginger ───────────────────────────────────────────
    'ginger':            'ginger',

    # ── gourd ────────────────────────────────────────────
    'gourd':             'gourd',
    'zucchini':          'gourd',
    'courgette':         'gourd',
    'squash':            'gourd',
    'bottle_gourd':      'gourd',
    'lauka':             'gourd',
    'ash_gourd':         'gourd',
    'kubhindo':          'gourd',
    'chayote':           'gourd',
    'iskus':             'gourd',
    'snake_gourd':       'gourd',
    'chichindo':         'gourd',
    'sponge_gourd':      'gourd',
    'ghiraula':          'gourd',
    'bitter_gourd':      'gourd',
    'karela':            'gourd',
    'pointed_gourd':     'gourd',
    'chuche_karela':     'gourd',

    # ── grape ────────────────────────────────────────────
    'grape':             'grape',
    'grapes':            'grape',
    'red_grapes':        'grape',
    'green_grapes':      'grape',

    # ── grapefruit ───────────────────────────────────────
    'grapefruit':        'grapefruit',
    'pomelo':            'grapefruit',

    # ── guacamole ────────────────────────────────────────
    'guacamole':         'guacamole',

    # ── juice ────────────────────────────────────────────
    'juice':             'juice',
    'fruit_juice':       'juice',

    # ── leafy_greens ─────────────────────────────────────
    'leafy_greens':      'leafy_greens',
    'lettuce':           'leafy_greens',
    'kale':              'leafy_greens',
    'chard':             'leafy_greens',
    'green_salad':       'leafy_greens',
    'salad':             'leafy_greens',
    'bethu_ko_saag':     'leafy_greens',
    'rayo_ko_saag':      'leafy_greens',
    'tori_ko_saag':      'leafy_greens',
    'moringa_leaves':    'leafy_greens',
    'stinging_nettle':   'leafy_greens',
    'sisnu':             'leafy_greens',
    'fiddlehead_ferns':  'leafy_greens',
    'niguro':            'leafy_greens',
    'garden_cress':      'leafy_greens',
    'chamsur_ko_saag':   'leafy_greens',
    'arugula':           'leafy_greens',
    'parsley':           'leafy_greens',
    'mint':              'leafy_greens',
    'basil':             'leafy_greens',

    # ── lemon (lime → lemon, нет отдельного класса) ──────
    'lemon':             'lemon',
    'lime':              'lemon',
    'nimbu':             'lemon',
    'kagati':            'lemon',

    # ── mango ────────────────────────────────────────────
    'mango':             'mango',
    'mangoes':           'mango',

    # ── melon (watermelon → melon) ────────────────────────
    'melon':             'melon',
    'watermelon':        'melon',
    'cantaloupe':        'melon',
    'honeydew':          'melon',

    # ── milk ─────────────────────────────────────────────
    'milk':              'milk',
    'dairy':             'milk',

    # ── mushroom ─────────────────────────────────────────
    'mushroom':          'mushroom',
    'mushrooms':         'mushroom',

    # ── mustard ──────────────────────────────────────────
    'mustard':           'mustard',
    'mustard_seeds':     'mustard',

    # ── noodles ──────────────────────────────────────────
    'noodles':           'noodles',
    'noodle':            'noodles',
    'pasta':             'noodles',
    'spaghetti':         'noodles',
    'farfalle_pasta':    'noodles',
    'fusilli_pasta':     'noodles',
    'chowmein_noodles':  'noodles',
    'thukpa_noodles':    'noodles',
    'ramen':             'noodles',

    # ── nuts ─────────────────────────────────────────────
    'nuts':              'nuts',
    'nut':               'nuts',
    'peanut':            'nuts',
    'peanuts':           'nuts',
    'cashew':            'nuts',
    'cashews':           'nuts',
    'almond':            'nuts',
    'almonds':           'nuts',
    'walnut':            'nuts',
    'walnuts':           'nuts',
    'pistachio':         'nuts',
    'hazelnut':          'nuts',

    # ── oil ──────────────────────────────────────────────
    'oil':               'oil',
    'olive_oil':         'oil',
    'vegetable_oil':     'oil',
    'sunflower_oil':     'oil',

    # ── olive ────────────────────────────────────────────
    'olive':             'olive',
    'olives':            'olive',
    'black_olive':       'olive',
    'green_olive':       'olive',

    # ── onion ────────────────────────────────────────────
    'onion':             'onion',
    'onions':            'onion',
    'red_onion':         'onion',
    'shallot':           'onion',

    # ── orange ───────────────────────────────────────────
    'orange':            'orange',
    'oranges':           'orange',

    # ── peach (apricot → peach, визуально похоже) ─────────
    'peach':             'peach',
    'peaches':           'peach',
    'apricot':           'peach',

    # ── pear ─────────────────────────────────────────────
    'pear':              'pear',
    'pears':             'pear',

    # ── peas ─────────────────────────────────────────────
    'peas':              'peas',
    'pea':               'peas',
    'green_peas':        'peas',
    'garden_peas':       'peas',
    'chickpeas':         'peas',
    'chickpea':          'peas',
    'edamame':           'peas',

    # ── pickles ──────────────────────────────────────────
    'pickles':           'pickles',
    'pickle':            'pickles',
    'gherkin':           'pickles',
    'kimchi':            'pickles',

    # ── pineapple ────────────────────────────────────────
    'pineapple':         'pineapple',

    # ── pomegrante (сохраняем оригинальную опечатку) ─────
    'pomegrante':        'pomegrante',
    'pomegranate':       'pomegrante',
    'granada':           'pomegrante',

    # ── pork (ham/sausage → pork) ────────────────────────
    'pork':              'pork',
    'pork_belly':        'pork',
    'ham':               'pork',
    'sausage':           'pork',
    'sausages':          'pork',
    'hot_dog':           'pork',
    'hotdog':            'pork',
    'salami':            'pork',
    'pepperoni':         'pork',

    # ── pot ──────────────────────────────────────────────
    'pot':               'pot',
    'pan':               'pot',
    'bowl':              'pot',
    'cooking_pot':       'pot',

    # ── potato (sweet_potato → potato) ───────────────────
    'potato':            'potato',
    'potatoes':          'potato',
    'sweet_potato':      'potato',
    'suthuni':           'potato',
    'yam':               'potato',

    # ── pudding (yogurt → pudding) ───────────────────────
    'pudding':           'pudding',
    'yogurt':            'pudding',
    'yoghurt':           'pudding',
    'custard':           'pudding',

    # ── pumpkin ──────────────────────────────────────────
    'pumpkin':           'pumpkin',
    'farsi':             'pumpkin',

    # ── raddish ──────────────────────────────────────────
    'raddish':           'raddish',
    'radish':            'raddish',
    'turnip':            'raddish',
    'beetroot':          'raddish',
    'beet':              'raddish',

    # ── rice ─────────────────────────────────────────────
    'rice':              'rice',
    'chamal':            'rice',
    'beaten_rice':       'rice',
    'chiura':            'rice',
    'rice_ball':         'rice',

    # ── salt ─────────────────────────────────────────────
    'salt':              'salt',

    # ── shrimp ───────────────────────────────────────────
    'shrimp':            'shrimp',
    'prawn':             'shrimp',
    'prawns':            'shrimp',

    # ── smoothie ─────────────────────────────────────────
    'smoothie':          'smoothie',
    'shake':             'smoothie',
    'milkshake':         'smoothie',

    # ── sour_cream ───────────────────────────────────────
    'sour_cream':        'sour_cream',
    'sour cream':        'sour_cream',
    'creme_fraiche':     'sour_cream',

    # ── spinach ──────────────────────────────────────────
    'spinach':           'spinach',
    'palak':             'spinach',
    'palungo':           'spinach',
    'indian_spinach':    'spinach',
    'nepali_spinach':    'spinach',

    # ── strawberry ───────────────────────────────────────
    'strawberry':        'strawberry',
    'strawberries':      'strawberry',

    # ── sugar (honey/jam → sugar) ─────────────────────────
    'sugar':             'sugar',

    # ── tangerine ────────────────────────────────────────
    'tangerine':         'tangerine',
    'mandarin':          'tangerine',
    'clementine':        'tangerine',

    # ── tomato (ketchup/sauce → tomato) ──────────────────
    'tomato':            'tomato',
    'tomatoes':          'tomato',
    'cherry_tomato':     'tomato',
}

def normalize(name: str) -> str:
    """Нормализует имя класса и ищет в SYNONYMS.
    Возвращает target-класс или None если дропать."""
    n = name.lower().replace(' ', '_').replace('-', '_')
    return SYNONYMS.get(n)  # None если не найден

def load_ds_classes(ds_path: str) -> list:
    yaml_files = list(Path(ds_path).glob('*.yaml'))
    if not yaml_files:
        return []
    with open(yaml_files[0]) as f:
        cfg = yaml.safe_load(f)
    names = cfg.get('names', [])
    # YOLO26 может хранить names как dict {0: 'cat', 1: 'dog', ...}
    if isinstance(names, dict):
        names = [names[i] for i in sorted(names)]
    return names

classes_ds1 = load_ds_classes('/kaggle/working/datasets/ds1')
classes_ds2 = load_ds_classes('/kaggle/working/datasets/ds2')
classes_ds3 = load_ds_classes('/kaggle/working/datasets/ds3')
classes_ds4 = load_ds_classes('/kaggle/working/datasets/ds4')

print(f'ds1: {len(classes_ds1)} классов — {classes_ds1[:6]}...')
print(f'ds2: {len(classes_ds2)} классов — {classes_ds2[:6]}...')
print(f'ds3: {len(classes_ds3)} классов — {classes_ds3[:6]}...')
print(f'ds4: {len(classes_ds4)} классов — {classes_ds4[:6]}...')

# Диагностика: какие классы датасетов НЕ покрыты синонимами
print('\n── Непокрытые классы (будут дропнуты) ──')
for name, cls_list in [('ds1', classes_ds1), ('ds2', classes_ds2),
                        ('ds3', classes_ds3), ('ds4', classes_ds4)]:
    dropped = [c for c in cls_list if normalize(c) is None]
    if dropped:
        print(f'  {name}: {dropped}')

with open('/kaggle/working/unified_classes.json', 'w') as f:
    json.dump(TARGET_CLASSES, f, indent=2, ensure_ascii=False)
print(f'\nИтого целевых классов: {len(TARGET_CLASSES)}')
print(TARGET_CLASSES)


In [ ]:
import shutil
from pathlib import Path
from collections import Counter

MERGED_DIR = Path('/kaggle/working/merged_dataset')
for split in ['train', 'valid', 'test']:
    (MERGED_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (MERGED_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

def build_remap(src_classes: list) -> dict:
    """Строит маппинг: старый_id → новый_id в TARGET_CLASSES.
    Классы, не найденные в SYNONYMS, не включаются (дропаются)."""
    remap = {}
    for i, name in enumerate(src_classes):
        target = normalize(name)
        if target is not None and target in TARGET_SET:
            remap[i] = TARGET_CLASSES.index(target)
    return remap

def copy_and_remap(ds_path: str, remap: dict, prefix: str) -> dict:
    """
    Копирует изображения и ремапит аннотации.
    Изображения БЕЗ аннотаций к целевым классам — НЕ копируются.
    Возвращает статистику.
    """
    ds_path = Path(ds_path)
    stats = Counter()

    for split_src in ['train', 'valid', 'test']:
        # поддержка 'val' как альтернативы 'valid'
        for split_try in [split_src, 'val' if split_src == 'valid' else None]:
            if split_try is None:
                continue
            img_dir = ds_path / split_try / 'images'
            lbl_dir = ds_path / split_try / 'labels'
            if not img_dir.exists():
                continue

            for img_file in img_dir.glob('*'):
                if img_file.suffix.lower() not in ('.jpg', '.jpeg', '.png', '.bmp'):
                    continue

                lbl_file = lbl_dir / (img_file.stem + '.txt')

                # Ремапим аннотации
                new_lines = []
                if lbl_file.exists():
                    with open(lbl_file) as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) >= 5:
                                old_cls = int(parts[0])
                                if old_cls in remap:
                                    new_cls = remap[old_cls]
                                    new_lines.append(
                                        f'{new_cls} ' + ' '.join(parts[1:])
                                    )

                # ── ФИЛЬТРАЦИЯ: пропускаем изображения без целевых классов ──
                if not new_lines:
                    stats['skipped'] += 1
                    continue

                new_name = f'{prefix}_{img_file.name}'
                shutil.copy(img_file,
                            MERGED_DIR / split_src / 'images' / new_name)

                lbl_dst = MERGED_DIR / split_src / 'labels' / (
                    Path(new_name).stem + '.txt'
                )
                lbl_dst.write_text('\n'.join(new_lines))
                stats['copied'] += 1
            break  # нашли нужную папку, выходим

    return stats

remap1 = build_remap(classes_ds1)
remap2 = build_remap(classes_ds2)
remap3 = build_remap(classes_ds3)
remap4 = build_remap(classes_ds4)

print('Объединяем датасеты...')
for ds_id, remap in [('ds1', remap1), ('ds2', remap2),
                      ('ds3', remap3), ('ds4', remap4)]:
    s = copy_and_remap(f'/kaggle/working/datasets/{ds_id}', remap, ds_id)
    print(f'  {ds_id}: скопировано={s["copied"]}, пропущено={s["skipped"]}')

print('\nИтого по сплитам:')
for split in ['train', 'valid', 'test']:
    n = len(list((MERGED_DIR / split / 'images').glob('*')))
    print(f'  {split}: {n} изображений')


In [ ]:
import yaml

data_yaml = {
    'path': '/kaggle/working/merged_dataset',
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc':    len(TARGET_CLASSES),
    'names': TARGET_CLASSES,
}

with open('/kaggle/working/merged_dataset/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print('data.yaml создан:')
with open('/kaggle/working/merged_dataset/data.yaml') as f:
    print(f.read())


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

class_counts = Counter()
for lbl_file in (MERGED_DIR / 'train' / 'labels').glob('*.txt'):
    with open(lbl_file) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                class_counts[int(parts[0])] += 1

print('Топ-20 классов (train):')
for idx, cnt in class_counts.most_common(20):
    print(f'  {TARGET_CLASSES[idx]:25s}: {cnt:5d}')

rare = [TARGET_CLASSES[i] for i in range(len(TARGET_CLASSES))
        if class_counts.get(i, 0) < 50]
print(f'\nРедкие классы (<50 аннотаций в train): {rare}')
print(f'Всего аннотаций в train: {sum(class_counts.values())}')

counts_sorted = sorted(
    [(TARGET_CLASSES[i], class_counts.get(i, 0))
     for i in range(len(TARGET_CLASSES))],
    key=lambda x: -x[1]
)
names_s, vals_s = zip(*counts_sorted)

fig, ax = plt.subplots(figsize=(max(16, len(TARGET_CLASSES) * 0.28), 6))
colors = ['#e74c3c' if v < 50 else '#f39c12' if v < 200 else '#27ae60'
          for v in vals_s]
ax.bar(names_s, vals_s, color=colors)
ax.set_xticklabels(names_s, rotation=90, fontsize=8)
ax.set_ylabel('Аннотаций')
ax.set_title('Распределение классов (красный<50, жёлтый<200, зелёный≥200)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/class_distribution.png', dpi=150)
plt.show()


In [ ]:
from ultralytics import YOLO

MODEL_NAME  = 'yolo26m.pt'   # yolo26s / yolo26l тоже работают
DATA_YAML   = '/kaggle/working/merged_dataset/data.yaml'
PROJECT_DIR = '/kaggle/working/runs'
RUN_NAME    = 'ingredients_yolo26m'
NUM_EPOCHS  = 100
IMGSZ       = 640    # 832 даёт +2% mAP, но медленнее
BATCH_SIZE  = 16     # T4 16 GB: yolo26m@640 влезает; при 832 → 8

model = YOLO(MODEL_NAME)

results = model.train(
    data          = DATA_YAML,
    epochs        = NUM_EPOCHS,
    imgsz         = IMGSZ,
    batch         = BATCH_SIZE,
    project       = PROJECT_DIR,
    name          = RUN_NAME,
    # ── Оптимизатор ──────────────────────────────────────
    optimizer     = 'AdamW',
    lr0           = 1e-3,
    lrf           = 0.01,      # cosine decay: lr_final = lr0 * lrf
    momentum      = 0.937,
    weight_decay  = 5e-4,
    warmup_epochs = 3,
    # ── Аугментации ──────────────────────────────────────
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    translate     = 0.1,
    scale         = 0.5,
    fliplr        = 0.5,
    mosaic        = 1.0,   # склейка 4 изображений — очень важно!
    mixup         = 0.3,   # помогает при дисбалансе классов
    copy_paste    = 0.3,   # добавляет примеры редких классов
    close_mosaic  = 10,    # отключить mosaic за 10 эпох до конца
    label_smoothing = 0.1, # устойчивость при шумных аннотациях
    # ── Прочее ───────────────────────────────────────────
    amp           = True,  # mixed precision (быстрее, меньше VRAM)
    cache         = 'ram', # кэш изображений: ускоряет ~1.5x
    device        = [0,1],
    workers       = 4,
    patience      = 30,    # early stopping
    save          = True,
    save_period   = 5,
    plots         = True,
    val           = True,
    verbose       = True
)

print(f'\nОбучение завершено!')
print(f'Лучшая модель: {PROJECT_DIR}/{RUN_NAME}/weights/best.pt')


In [ ]:
from ultralytics import YOLO

BEST_MODEL = f'/kaggle/working/runs/{RUN_NAME}/weights/best.pt'
model = YOLO(BEST_MODEL)

metrics = model.val(
    data    = DATA_YAML,
    imgsz   = IMGSZ,
    batch   = BATCH_SIZE,
    device  = 0,
    verbose = True,
    plots   = True,
)

print('\\n=== ИТОГОВЫЕ МЕТРИКИ ===')
print(f'mAP@0.5:      {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')
print(f'Precision:    {metrics.box.mp:.4f}')
print(f'Recall:       {metrics.box.mr:.4f}')

print('\\nМетрики по классам (mAP@0.5):')
for cls_name, ap in zip(UNIFIED_CLASSES, metrics.box.ap50):
    bar = '█' * int(ap * 20)
    print(f'  {cls_name:25s} {ap:.3f} {bar}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

results_csv = Path(f'/kaggle/working/runs/{RUN_NAME}/results.csv')
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    def safe_plot(ax, col, title, color='blue'):
        if col in df.columns:
            ax.plot(df[col], color=color)
            ax.set_title(title)
            ax.set_xlabel('Epoch')
            ax.grid(alpha=0.3)

    safe_plot(axes[0,0], 'train/box_loss',      'Train Box Loss',    'blue')
    safe_plot(axes[0,1], 'train/cls_loss',      'Train Cls Loss',    'orange')
    safe_plot(axes[0,2], 'train/dfl_loss',      'Train DFL Loss',    'purple')
    safe_plot(axes[1,0], 'metrics/mAP50(B)',    'mAP@0.5',           'green')
    safe_plot(axes[1,1], 'metrics/mAP50-95(B)', 'mAP@0.5:0.95',      'darkgreen')
    safe_plot(axes[1,2], 'val/box_loss',        'Val Box Loss',      'red')

    plt.tight_layout()
    plt.savefig('/kaggle/working/training_curves.png', dpi=150)
    plt.show()

    if 'metrics/mAP50(B)' in df.columns:
        best_epoch = df['metrics/mAP50(B)'].idxmax()
        print(f'Лучшая эпоха:  {best_epoch + 1}')
        print(f'mAP@0.5:      {df["metrics/mAP50(B)"].max():.4f}')

In [ ]:
import urllib.request
import matplotlib.pyplot as plt
from ultralytics import YOLO

BEST_MODEL = f'/kaggle/working/runs/{RUN_NAME}/weights/best.pt'
model = YOLO(BEST_MODEL)

TEST_URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/6/6d/Good_Food_Display_-_NCI_Visuals_Online.jpg/1200px-Good_Food_Display_-_NCI_Visuals_Online.jpg'
urllib.request.urlretrieve(TEST_URL, '/kaggle/working/test_food.jpg')

results = model(
    source  = '/kaggle/working/test_food.jpg',
    conf    = 0.25,
    iou     = 0.45,
    imgsz   = IMGSZ,
    save    = True,
    project = '/kaggle/working',
    name    = 'inference_test',
    verbose = False,
)

r = results[0]
print(f'Найдено объектов: {len(r.boxes)}')
for box in r.boxes:
    cls_id = int(box.cls[0])
    conf   = float(box.conf[0])
    name   = UNIFIED_CLASSES[cls_id] if cls_id < len(UNIFIED_CLASSES) else f'cls_{cls_id}'
    print(f'  {name:25s}: conf={conf:.3f}')

In [ ]:
# from ultralytics import YOLO

# LAST_CKPT = f'/kaggle/working/runs/{RUN_NAME}/weights/last.pt'
# model = YOLO(LAST_CKPT)
# results = model.train(
#     resume  = True,   # продолжает с последнего checkpoint
#     epochs  = 150,    # общее число (не дополнительных)
#     device  = 0,
# )

In [ ]:
# Раскомментируйте нужный блок:

# YOLOv11m — новее (2024), ~5% лучше v8m при той же скорости
# model = YOLO('yolo11m.pt')

# RT-DETR-L — трансформер, лучше на сложных сценах
# model = YOLO('rtdetr-l.pt')
# model.train(
#     data         = DATA_YAML,
#     epochs       = 80,
#     imgsz        = 640,
#     batch        = 8,       # меньше для RT-DETR
#     lr0          = 1e-4,
#     optimizer    = 'AdamW',
#     warmup_epochs= 3,
#     patience     = 20,
#     project      = PROJECT_DIR,
#     name         = 'ingredients_rtdetr',
#     device       = 0,
#     plots        = True,
# )

print("Раскомментируйте нужный блок для запуска.")

In [ ]:
import shutil
import json
from pathlib import Path
from ultralytics import YOLO

BEST_MODEL = f'/kaggle/working/runs/{RUN_NAME}/weights/best.pt'
OUTPUT_DIR = Path('/kaggle/working/ingredient_detector')
OUTPUT_DIR.mkdir(exist_ok=True)

shutil.copy(BEST_MODEL, OUTPUT_DIR / 'best.pt')
shutil.copy('/kaggle/working/merged_dataset/data.yaml', OUTPUT_DIR / 'data.yaml')
shutil.copy('/kaggle/working/unified_classes.json',     OUTPUT_DIR / 'unified_classes.json')

# Экспорт в ONNX (инференс без PyTorch, быстрее на CPU)
model = YOLO(BEST_MODEL)
model.export(format='onnx', imgsz=IMGSZ, opset=17, simplify=True)
onnx_src = BEST_MODEL.replace('.pt', '.onnx')
if Path(onnx_src).exists():
    shutil.copy(onnx_src, OUTPUT_DIR / 'best.onnx')

for fname in ['training_curves.png', 'class_distribution.png']:
    src = Path(f'/kaggle/working/{fname}')
    if src.exists():
        shutil.copy(src, OUTPUT_DIR / fname)

print(f'Готово: {OUTPUT_DIR}')
for f in sorted(OUTPUT_DIR.iterdir()):
    size = f.stat().st_size / 1e6
    print(f'  {f.name}: {size:.1f} MB')

In [ ]:
# from ultralytics import YOLO
# import json

# class IngredientDetector:
#     """
#     Обёртка для интеграции в кулинарного ассистента.
#     Принимает путь к изображению, возвращает список ингредиентов.
#     """
#     def __init__(self, model_path, classes_path, conf=0.25, iou=0.45):
#         self.model = YOLO(model_path)
#         self.conf  = conf
#         self.iou   = iou
#         with open(classes_path) as f:
#             self.classes = json.load(f)

#     def detect(self, image_path):
#         results = self.model(
#             image_path, conf=self.conf, iou=self.iou, verbose=False
#         )[0]

#         detected = []
#         for box in results.boxes:
#             cls_id = int(box.cls[0])
#             detected.append({
#                 'ingredient': self.classes[cls_id],
#                 'confidence': round(float(box.conf[0]), 3),
#                 'bbox': [round(v) for v in box.xyxy[0].tolist()],
#             })

#         # Дедупликация: берём детекцию с максимальной confidence
#         unique = {}
#         for d in detected:
#             name = d['ingredient']
#             if name not in unique or d['confidence'] > unique[name]['confidence']:
#                 unique[name] = d

#         return {
#             'ingredients': sorted(unique.values(), key=lambda x: -x['confidence']),
#             'ingredient_names': sorted(unique.keys()),  # для LLM
#         }


# # Демо
# detector = IngredientDetector(
#     model_path   = '/kaggle/working/ingredient_detector/best.pt',
#     classes_path = '/kaggle/working/ingredient_detector/unified_classes.json',
# )

# result = detector.detect('/kaggle/working/test_food.jpg')

# print('Обнаруженные ингредиенты:')
# for ing in result['ingredients']:
#     bar = '▓' * int(ing['confidence'] * 20)
#     print(f"  {ing['ingredient']:20s} {ing['confidence']:.2f} {bar}")

# print(f"\\nДля LLM-ассистента:")
# print(f"  ingredients = {result['ingredient_names']}")
# # Далее передаёте ingredient_names в промпт кулинарного ассистента